# Basic workflow for making a "surrogate" model that learns the unpolarized cross-section as function of $x_{\text{B}}$, $t$, $Q^{2}$, and $\phi$.

## (1): Import Libraries

### (1.1): Import Native Libraries:

In [ ]:
import datetime
import gc

### (1.2): Import 3rd-Party Libraries:

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.model_selection import train_test_split

### (1.3): Library Versions:

In [ ]:
print(f"[INFO]: numpy version: {np.__version__}")
print(f"[INFO]: pandas version: {pd.__version__}")
print(f"[INFO]: tensorflow version: {tf.__version__}")

### (1.4): Versioning:

In [ ]:
VERSION_NUMBER = 4
MINOR_NUMBER = 1
MAJOR_MINOR_NUMBER = f"{VERSION_NUMBER}_{MINOR_NUMBER}"

print(f"We are saving figures and data with the following appendage: {MAJOR_MINOR_NUMBER}")

## (2): Plotting Styles:

In [ ]:
plt.rcParams.update({"text.usetex": True, "font.family": "serif"})
plt.rcParams['xtick.direction'] = 'in'
plt.rcParams['xtick.major.size'] = 8.5
plt.rcParams['xtick.major.width'] = 0.5
plt.rcParams['xtick.minor.size'] = 3.5
plt.rcParams['xtick.minor.width'] = 0.5
plt.rcParams['xtick.minor.visible'] = True
plt.rcParams['xtick.top'] = True
plt.rcParams['xtick.labelsize'] = 15
plt.rcParams['ytick.direction'] = 'in'
plt.rcParams['ytick.major.size'] = 8.5
plt.rcParams['ytick.major.width'] = 0.5
plt.rcParams['ytick.minor.size'] = 3.5
plt.rcParams['ytick.minor.width'] = 0.5
plt.rcParams['ytick.minor.visible'] = True
plt.rcParams['ytick.right'] = True
plt.rcParams['ytick.labelsize'] = 14
plt.rcParams['savefig.dpi'] = 300

## (3): Data Loading:

### (3.1): Loading Main File:

In [ ]:
test_dataframe = pd.read_csv(f'./local/version_{MAJOR_MINOR_NUMBER}/data/refined_cross_section_data_v{MAJOR_MINOR_NUMBER}.csv')

### (3.2): Loading in the supervised learning $(x, y)$ pairs:

In [ ]:
x_data = test_dataframe[["k", "t", "x_b", "q_squared", "phi"]]
y_data = test_dataframe[["unp_beam_unp_target_xsec"]]

### (3.3): Checking out the $x$ data:

In [ ]:
x_data.head()

### (3.4): Checking out the $y$ data:

In [ ]:
y_data.head()

### (3.5): Splitting along training/validation/testing:

In [ ]:
x_remaining, x_testing, y_remaining, y_testing = train_test_split(
    x_data, y_data, test_size = 0.1, shuffle = True, random_state = 7008)

x_training, x_validation, y_training, y_validation = train_test_split(
    x_remaining, y_remaining, test_size = 0.1, shuffle = True, random_state = 7008)

In [ ]:
len(x_training)

In [ ]:
len(x_validation)

In [ ]:
len(x_testing)

## (4): DNN Stuff:

### (4.1): MSE Loss:

[NOTE]: If the cross-section data is not normalized, then this loss may bias several kinds of cross-sections because it treats all those values on about the same footing (i.e. no weighting), but all those values vary from 0.01 to 1.0 to 10.0 sometimes (in respective units).

In [ ]:
class CrossSectionLoss(tf.keras.losses.Loss):
    def call(self, y_true, y_pred):
        return tf.reduce_mean(tf.square(y_true - y_pred))

### (4.2): DNN Architecture:

In [ ]:
class CrossSectionSurrogateModel(tf.keras.Model):

    # https://keras.io/api/models/model/ -> follow this for custom model architecture

    def __init__(self, symmetry_loss_weight = 1.0):
        super().__init__()

        self.symmetry_loss_weight = symmetry_loss_weight

        # self.data_loss_tracker = tf.keras.metrics.Mean(name = "data_loss")
        # self.symmetry_loss_tracker = tf.keras.metrics.Mean(name = "symmetry_loss")

        initializer = tf.keras.initializers.GlorotNormal(seed = None)

        self.dense_layer_1 = tf.keras.layers.Dense(64, kernel_initializer = initializer, activation = "relu")
        self.dense_layer_2 = tf.keras.layers.Dense(64, kernel_initializer = initializer, activation = "relu")
        self.dense_layer_3 = tf.keras.layers.Dense(64, kernel_initializer = initializer, activation = "relu")
        self.dense_layer_4 = tf.keras.layers.Dense(64, kernel_initializer = initializer, activation = "relu")

        # linear activation is default activation if `activation` key is not specified: https://www.tensorflow.org/api_docs/python/tf/keras/layers/Dense
        self.cross_section_output = tf.keras.layers.Dense(1, activation = "linear", name = "cross_section")

        # custom loss business:
        self.cross_section_loss_tracker = CrossSectionLoss()

    def azimuthal_symmetry_loss(self, X_batch, training = True):

        X_plus = X_batch

        phi = X_batch[:, -1]

        X_minus = tf.concat([X_batch[:, :-1], tf.expand_dims(-phi, axis = 1)], axis = 1)

        y_plus = self(X_plus, training = training)
        y_minus = self(X_minus, training = training)

        return tf.reduce_mean(tf.square(y_plus - y_minus))

    def call(self, inputs, training = False):

        # hidden layer computation:
        hidden_layer = self.dense_layer_1(inputs)
        hidden_layer = self.dense_layer_2(hidden_layer)
        hidden_layer = self.dense_layer_3(hidden_layer)
        hidden_layer = self.dense_layer_4(hidden_layer)
        cross_section_output = self.cross_section_output(hidden_layer)

        return cross_section_output
    
    def train_step(self, data):

        # unpack data:
        X_batch_data, y_batch_data = data

        with tf.GradientTape() as tape:
            # forward pass:
            predictions = self(X_batch_data, training = True)

            # recall: `Instead, use `model.compute_loss(x, y, y_pred, sample_weight)`
            data_loss = self.compute_loss(X_batch_data, y_batch_data, predictions)

            # compute phi symmetry loss:
            symmetry_loss = self.azimuthal_symmetry_loss(X_batch_data, training = True)

            # total loss is just a weighted sum:
            total_loss = data_loss + self.symmetry_loss_weight * symmetry_loss

        gradients = tape.gradient(total_loss, self.trainable_variables)
        self.optimizer.apply_gradients(zip(gradients,self.trainable_variables))

        for metric in self.metrics:
            if metric.name == "loss":
                metric.update_state(total_loss)
            else:
                metric.update_state(y_batch_data, predictions)

        return {
            "loss": total_loss,
            "data_loss": data_loss,
            "symmetry_loss": symmetry_loss,
            **{m.name: m.result() for m in self.metrics}
        }

    def test_step(self, data):
        # unpack data:
        X_batch_data, y_batch_data = data
        
        # forward pass evaluation:
        predictions = self(X_batch_data, training = False)

        # recall: `Instead, use `model.compute_loss(x, y, y_pred, sample_weight)`
        data_loss = self.compute_loss(X_batch_data, y_batch_data, predictions)

         # compute phi symmetry loss:
        symmetry_loss = self.azimuthal_symmetry_loss(X_batch_data, training = False)

        # total loss is just a weighted sum:
        total_loss = data_loss + self.symmetry_loss_weight * symmetry_loss

        for metric in self.metrics:
            if metric.name == "loss":
                metric.update_state(total_loss)
            else:
                metric.update_state(y_batch_data, predictions)
            
        return {
            "loss": total_loss,
            "data_loss": data_loss,
            "symmetry_loss": symmetry_loss,
            **{m.name: m.result() for m in self.metrics}
        }

## (5): **Actually Fitting the Model**:

### (5.1): Hyperparameters-ish:

In [ ]:
NUMBER_OF_REPLICAS = 1
BASE_LEARNING_RATE = 3e-4
TOTAL_EPOCHS = 2000
SYMMETRY_LOSS_PARAMETER = 1.0
BATCH_SIZE = 8

### **(5.2): The Fit Routine!**

In [ ]:
all_histories = []
all_point_predictions = []
models = []

for index in range(NUMBER_OF_REPLICAS):
    replica_number = index + 1
    print(f"[INFO]: Now training replica #{replica_number}")

    tf.keras.backend.clear_session()
    gc.collect()

    dnn_model = CrossSectionSurrogateModel(symmetry_loss_weight = SYMMETRY_LOSS_PARAMETER)
    dnn_model.compile(
        # LR is alpha in ADAM, which is stepsize:
        optimizer = tf.keras.optimizers.Adam(BASE_LEARNING_RATE),
        loss = CrossSectionLoss())

    dnn_model_history = dnn_model.fit(
        x_training, y_training,
        validation_data = (x_validation, y_validation),
        epochs = TOTAL_EPOCHS,
        # [NOTE]: BATCHSIZE really matters!
        # batch_size = len(x_training),
        batch_size = BATCH_SIZE,
        callbacks = [
            tf.keras.callbacks.ReduceLROnPlateau(
                monitor = "val_loss", factor = 0.5, patience = 50, min_lr = 1e-6,
                verbose = 0),
        ],
        verbose = 0)
    
    all_histories.append(dnn_model_history.history)

    # evaluating the DNN:
    model_testing_evaluation_metrics = dnn_model.evaluate(x_testing, y_testing, verbose = 0)
    print(f"[INFO]: Evaluation metrics are: {model_testing_evaluation_metrics}")

    dictionary_of_keras_metrics = dict(zip(dnn_model.metrics_names, model_testing_evaluation_metrics))
    model_testing_loss = dictionary_of_keras_metrics["loss"]
    print(f"[INFO]: Test loss for replica #{replica_number}: {model_testing_loss}")

    # loss plots/learning curves:
    figure, axis = plt.subplots(1, 1, figsize = (6, 6))

    axis.plot(dnn_model_history.history['loss'],
        label = "Training Loss", color = 'orange', alpha = 0.6)
    axis.plot(dnn_model_history.history['val_loss'],
        label = "Validation Loss", color = 'purple', alpha = 0.6)

    axis.set_xlabel(r"Epoch", fontsize = 14.)
    axis.set_ylabel(r"Loss", fontsize = 14.)
    axis.set_title(
        rf"Cross-Section Surrogate Model (Testing = ${model_testing_loss:.3f}$)",
        fontsize = 18.)
    axis.legend(fontsize = 14.)
    axis.grid(visible = False)

    axis.text(
        0.00, -0.11,
        f"Figure rendered {datetime.datetime.now().strftime('%Y%m%d-%H%M%S')}", 
        transform = axis.transAxes, verticalalignment = 'top',  horizontalalignment = 'left', fontsize = 9.,)

    for extension in ['png', 'eps']:
        figure.savefig(
            fname = f"./local/version_{MAJOR_MINOR_NUMBER}/learning_curves/xsec_surrogate_lc_replica_{replica_number}_v{MAJOR_MINOR_NUMBER}.{extension}",
            facecolor = 'white',
            transparent = False)

    plt.close(figure)

    del figure
    del axis

    #  log losses
    figure, axis = plt.subplots(1, 1, figsize = (6, 6))

    axis.plot(
        np.arange(0, TOTAL_EPOCHS, 1),
        np.log(dnn_model_history.history['loss']), linewidth = 1.0,
        label = "Log Training Loss", color = 'orange', alpha = 0.6)
    axis.plot(
        np.arange(0, TOTAL_EPOCHS, 1),
        np.log(dnn_model_history.history['val_loss']), linewidth = 1.0,
        label = "Log Validation Loss", color = 'purple', alpha = 0.6)

    axis.set_xlabel(r"Epoch", fontsize = 14.)
    axis.set_ylabel(r"Log (Loss)", fontsize = 14.)
    axis.set_title(
        rf"Cross-Section Surrogate Model (Log Testing = ${np.log(model_testing_loss):.3f}$)",
        fontsize = 18.)
    axis.legend(fontsize = 14.)
    axis.grid(visible = False)

    axis.text(
        0.00, -0.11,
        f"Figure rendered {datetime.datetime.now().strftime('%Y%m%d-%H%M%S')}", 
        transform = axis.transAxes, verticalalignment = 'top',  horizontalalignment = 'left', fontsize = 9.,)

    for extension in ['png', 'eps']:
        figure.savefig(
            fname = f"./local/version_{MAJOR_MINOR_NUMBER}/learning_curves/cross_section_surrogate_log_lc_replica_{replica_number}_v{MAJOR_MINOR_NUMBER}.{extension}",
            facecolor = 'white',
            transparent = False)

    plt.close(figure)

    del figure
    del axis
    
    # predictions at actual datapoints
    replica_point_predictions = dnn_model.predict(x_data, verbose = 0)
    all_point_predictions.append(replica_point_predictions)

    # the models are in memory...
    models.append(dnn_model)
    dnn_model.save(
        filepath = f"./local/version_{MAJOR_MINOR_NUMBER}/replicas/replica_{replica_number}_v{MAJOR_MINOR_NUMBER}.keras",
    )

all_point_predictions = np.array(all_point_predictions)

## (6): Making Plots of Every Local Fit to the Available Data:

### (6.1): Group the Dataframe by the Kinematic Settings:

In [ ]:
grouped = test_dataframe.groupby(['k', 't', 'x_b', 'q_squared'])

### (6.2): Make the Statistical Distribution Plots at Every Kinematic Setting:

In [ ]:
average_prediction = np.mean(all_point_predictions, axis = 0)
standard_dev_prediction = np.std(all_point_predictions, axis = 0)

# this is trento convention: -pi to pi:
phi_smooth = np.linspace(-np.pi, np.pi, 361)
special_phis = [0, np.pi/2., -np.pi/2., np.pi]

for (k_value, t_value, xb_value, qsquared_value), group in grouped:
    print(f"[INFO]: Processing k = {k_value}, t = {t_value}, xb = {xb_value}, Q2 = {qsquared_value}")

    group = group.sort_values('phi')

    xsec_err = group['unp_beam_unp_target_xsec_err'].values

    indices = group.index.values

    xsec_pred = average_prediction[indices, 0]
    xsec_std = standard_dev_prediction[indices, 0]

    x_smooth = np.column_stack([
        np.full_like(phi_smooth, k_value),
        np.full_like(phi_smooth, t_value),
        np.full_like(phi_smooth, xb_value),
        np.full_like(phi_smooth, qsquared_value),
        phi_smooth
    ])

    smooth_preds_all = np.array([ model.predict(x_smooth, verbose = 0) for model in models ])

    smooth_mean = np.mean(smooth_preds_all, axis = 0)
    smooth_std = np.std(smooth_preds_all, axis = 0)

    xsec_smooth_mean = smooth_mean[:, 0]
    xsec_smooth_std = smooth_std[:, 0]

    for phi_target in special_phis:
        phi_index = np.argmin(np.abs(phi_smooth - phi_target))
        phi_actual = phi_smooth[phi_index]
        sigma_value = xsec_smooth_std[phi_index]
        # print(
        #     f"[INFO]: "
        #     f"(xb = {xb_value:.3f}, t = {t_value:.3f}, Q2 = {qsquared_value:.3f}) "
        #     f"cross-section 1σ uncertainty at phi = {phi_actual:.3f} rad is ±{sigma_value:.6f}"
        # )

    phi = group['phi'].values
    xsec_actual = group['unp_beam_unp_target_xsec'].values

    xsec_res = xsec_actual - xsec_pred
    chi2_xsec = np.sum(xsec_res**2) / len(phi)

    residuals_figure, axes = plt.subplots(2, 1, figsize = (10, 8), sharex = 'col', layout = "tight")

    axes[1].text(
        -0.1, -0.1,
        fr"Figure rendered {datetime.datetime.now().strftime('%Y%m%d-%H%M%S')}", 
        transform = axes[1].transAxes)

    axes[0].plot(phi_smooth, xsec_smooth_mean, color = 'red', lw = 2, label = rf'Replica Average ($N = {NUMBER_OF_REPLICAS}$)')
    axes[0].fill_between(
        phi_smooth, xsec_smooth_mean - xsec_smooth_std, xsec_smooth_mean + xsec_smooth_std,
        color = 'red', alpha = 0.3,
        label = r'$\sigma$ band')
    
    axes[0].errorbar(
        phi, xsec_actual, yerr = xsec_err,
        fmt = 'o', mfc = 'white', mec = 'black', ms = 5, ecolor = 'black', elinewidth = 1, capsize = 2, alpha = 0.8,
        label = 'Experimental Data')
    axes[0].set_ylabel(r"$d^{4}\sigma$ [nb / GeV$^{4}$]", fontsize = 16.)
    axes[0].set_title(rf"Cross Section ($\chi^2_\nu = {chi2_xsec:.7f}$)", fontsize = 18.)
    axes[0].legend(fontsize = 14.)
    axes[0].grid(True, linestyle = ':', alpha = 0.6)

    axes[1].scatter(phi, xsec_res, color = 'blue', alpha = 0.6)
    axes[1].axhline(0, color = 'black', linestyle = '--')
    axes[1].set_xlabel(r"$\phi$ (radians)", fontsize = 16.)
    axes[1].set_title("Residuals", fontsize = 18.)
    axes[1].grid(True, linestyle = ':', alpha = 0.6)
    
    residuals_figure.suptitle(
        "Kinematic Setting:\n"
        rf"$k = {k_value}$, $t = {t_value}$, $x_\textrm{{B}} = {xb_value}$, $Q^2 = {qsquared_value}$",
        fontsize = 16.
    )

    filename = f"./local/version_{MAJOR_MINOR_NUMBER}/plots/k{k_value}_t{t_value}_xb{xb_value}_q2{qsquared_value}_residuals_v{MAJOR_MINOR_NUMBER}"
    
    for extension in ['png', 'eps']:
        residuals_figure.savefig(
            fname = f"{filename}.{extension}",
            facecolor = 'white', transparent = False)

    plt.close(residuals_figure)

In [ ]:
unique_k_values = test_dataframe.groupby(['k'])
print(f"[INFO]: there are {unique_k_values.ngroups} unique values for k.")

In [ ]:
for k_value, k_group in unique_k_values:
    xb_q2_groups = k_group.groupby(['x_b', 'q_squared'])
    xb_q2_combinations = xb_q2_groups.ngroups
    print(f"[INFO]: found a total of {xb_q2_combinations} (xb, Q^2) valuesat k = {k_value}")

In [ ]:
xb_q2_groups = test_dataframe.groupby(['x_b', 'q_squared'])
n_xb_q2 = xb_q2_groups.ngroups
print(n_xb_q2)

In [ ]:
for (xb_value, qsquared_value), group in xb_q2_groups:

    group = group.sort_values(['t', 'phi'])

    t_values = np.sort(group['t'].unique())

    phi_meshgrid, t_meshgrid = np.meshgrid(
        # this is a dense grid of phi values:
        np.linspace(-np.pi, np.pi, 361),
        t_values)

    phi_data = group['phi'].values
    t_data = group['t'].values

    indices = group.index.values

    xsec_pred = average_prediction[indices, 0]

    xsec_actual = group['unp_beam_unp_target_xsec'].values
    bsa_actual = group['unp_target_bsa'].values

    xsec_res = xsec_actual - xsec_pred

    colors_xsec = np.where(xsec_res >= 0, 'red', 'blue')

    model_surface_input = np.column_stack([
        t_meshgrid.ravel(),
        np.full(t_meshgrid.size, xb_value),
        np.full(t_meshgrid.size, qsquared_value),
        phi_meshgrid.ravel()
    ])

    surface_preds_all = np.array([
        model.predict(model_surface_input) for model in models
    ])

    surface_mean = np.mean(surface_preds_all, axis = 0)
    surface_std_dev = np.std(surface_preds_all, axis = 0)

    xsec_surface = surface_mean[:, 0].reshape(t_meshgrid.shape)

    xsec_stddev_surface = surface_std_dev[:, 0].reshape(t_meshgrid.shape)

    zero_plane_xsec = np.zeros_like(xsec_surface)

    fig = plt.figure(figsize = (14, 7), layout = "tight")

    ax1 = fig.add_subplot(1, 2, 1, projection = '3d')
    ax2 = fig.add_subplot(1, 2, 2, projection = '3d')

    # [NOTE]: this order actually determines some z-ordering stuff...
    ax1.plot_surface(
        phi_meshgrid, t_meshgrid, xsec_surface + xsec_stddev_surface,
        color = "gray", alpha = 0.30)
    ax1.plot_surface(
        phi_meshgrid, t_meshgrid, xsec_surface - xsec_stddev_surface,
        color = "gray", alpha = 0.30)
    ax1.plot_surface(
        phi_meshgrid, t_meshgrid, xsec_surface,
        cmap = 'viridis', alpha = 0.30)
    ax1.scatter(
        phi_data, t_data, xsec_actual, 
        facecolors = 'white', edgecolors = 'black', s = 20, linewidths = 0.5, alpha = 1.0)

    ax1.set_xlabel(r'$\phi$ [Radians]',
                labelpad = 16, fontsize = 16.)
    ax1.set_ylabel(r'$t$ [GeV$^{2}$]',
                labelpad = 16, fontsize = 16.)
    ax1.set_zlabel(r'$d^{4}\sigma^{UU}$ [nb GeV$^{-4}$]',
                labelpad = 7, fontsize = 16.)
    ax1.set_title('Cross Section', fontsize = 18.)

    ax2.plot_surface(
        phi_meshgrid, t_meshgrid, zero_plane_xsec,
        color = 'gray', alpha = 0.15)
    ax2.scatter(
        phi_data, t_data, xsec_res,
        color = colors_xsec, s = 20)

    ax2.set_xlabel(r'$\phi$ [Radians]',
                labelpad = 16, fontsize = 16.)
    ax2.set_ylabel(r'$t$ [GeV$^{2}$] ',
                labelpad = 16, fontsize = 16.)
    ax2.set_zlabel('Residuals',
                labelpad = 7, fontsize = 16.)
    ax2.set_title('Cross Section Residuals', fontsize = 18)

    fig.suptitle(
        r"DNN Interpolations Across $t$ and $\phi$"
        "\n"
        rf"Kinematic Setting: $x_\textrm{{B}} = {xb_value}$, $Q^2 = {qsquared_value}$ GeV$^{{2}}$",
        fontsize = 16)

    plot_filename = f"./local/version_{MAJOR_MINOR_NUMBER}/plots/surface_xb{xb_value}_q2{qsquared_value}_v{MAJOR_MINOR_NUMBER}"
    
    for extension in ['png', 'eps']:
        fig.savefig(f"{plot_filename}.{extension}", facecolor = 'white')

    plt.close(fig)

    # cleanup:
    del fig
    del ax1
    del ax2